# Konfigurasi Lokasi File ZIP

In [1]:
import os
os.environ["ZIP_PATH"] = "/content/dataset_v2_frozen.zip"

# 1 Mount Drive, Ekstrak ZIP, & Verifikasi Hash Manifest

In [2]:
import os, sys, re, json, hashlib, zipfile
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

BASE = os.environ.get("DATA_DIR", "/content/data_colab")
ZIP_PATH = os.environ.get("ZIP_PATH", "/content/dataset_v2_frozen.zip")

# Langsung ekstrak tanpa perlu mount Google Drive
if not os.path.exists(os.path.join(BASE, "train.csv")):
    zipfile.ZipFile(ZIP_PATH).extractall(BASE)

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(1 << 20), b""): h.update(ch)
    return h.hexdigest().lower()

m = pd.read_csv(f"{BASE}/MANIFEST_sha256.csv").set_index("berkas").sha256.str.lower()
FILES = ["train.csv", "val.csv", "dev_test.csv", "eval_benar.csv", "eval_kompas_cekfakta.csv", "text_pipeline.py"]
ok = {f: (os.path.exists(f"{BASE}/{f}") and sha(f"{BASE}/{f}") == m.get(f)) for f in FILES}

for f, v in ok.items():
    print("OK  " if v else "BEDA", f)

assert all(ok.values()), "Ada berkas yang tidak cocok dengan manifest: JANGAN lanjut."
assert not os.path.exists(f"{BASE}/LOCKBOX") and not any("lockbox" in x.lower() for x in os.listdir(BASE)), "LOCKBOX tidak boleh ada di sini"

sys.path.insert(0, BASE)
import text_pipeline as tp
assert tp.PIPELINE_VERSION == "tp-1.6"
print("\n>> VERIFIKASI BERHASIL: Lingkungan Colab bersih dan steril.")

OK   train.csv
OK   val.csv
OK   dev_test.csv
OK   eval_benar.csv
OK   eval_kompas_cekfakta.csv
OK   text_pipeline.py

>> VERIFIKASI BERHASIL: Lingkungan Colab bersih dan steril.


# 2 Muat Dataset & Fungsi Utilitas Statistik

In [3]:
tr, va, te = [pd.read_csv(f"{BASE}/{s}.csv") for s in ("train", "val", "dev_test")]
for d in (tr, va, te):
    for c in ("text", "title", "body"): d[c] = d[c].fillna("").astype(str)

print("Jumlah baris:", {k: len(v) for k, v in dict(train=tr, val=va, dev_test=te).items()})
ytr, yva, yte = tr.label.to_numpy(), va.label.to_numpy(), te.label.to_numpy()

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h

def fmt(k, n):
    if n == 0: return "n=0"
    lo, hi = wilson(k, n); return f"{k/n*100:5.1f}% [{max(lo,0)*100:4.1f}-{min(hi,1)*100:4.1f}] (n={n})"

def prep_title(t):
    return tp.ensure_end(tp.norm_title(str(t)))

def run(Xtr, y, Xev, seed=0):
    v = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    clf = LogisticRegression(max_iter=2000, random_state=seed).fit(v.fit_transform(Xtr), y)
    return v, clf, clf.predict(v.transform(Xev))

RESULTS = {}

Jumlah baris: {'train': 1660, 'val': 236, 'dev_test': 236}


# 3 Uji Label Diacak (Permutation Test / Leak Detector)

In [4]:
accs = []
for s in range(20):
    ys = np.random.default_rng(s).permutation(ytr)
    _, _, p = run(tr.text, ys, va.text); accs.append((p == yva).mean() * 100)

print(f"Label diacak, val: rerata {np.mean(accs):.1f}% | sd {np.std(accs):.1f} | maks {np.max(accs):.1f}%  (harapan ≈ 50 ± 3)")
RESULTS["permutasi"] = dict(mean=float(np.mean(accs)), sd=float(np.std(accs)), max=float(np.max(accs)))

if np.mean(accs) > 55:
    print(">> PERINGATAN: rerata di atas 55%. Periksa duplikat/kebocoran antara train dan val sebelum lanjut.")
else:
    print(">> STATUS: LULUS (Rerata dalam kisaran acak).")
print("Catatan: uji ini TIDAK menguji confound sumber/genre. Lulus != model membaca substansi.")

Label diacak, val: rerata 49.4% | sd 3.7 | maks 55.9%  (harapan ≈ 50 ± 3)
>> STATUS: LULUS (Rerata dalam kisaran acak).
Catatan: uji ini TIDAK menguji confound sumber/genre. Lulus != model membaca substansi.


# 4: Baseline TF-IDF + Logistic Regression (Teks Penuh)

In [5]:
v, clf, pv = run(tr.text, ytr, va.text)
pt = clf.predict(v.transform(te.text))

for nm, y, p in (("val", yva, pv), ("dev_test", yte, pt)):
    k = int((p == y).sum()); n = len(y)
    print(f"{nm:<9} akurasi {fmt(k, n)} | F1 hoaks {f1_score(y, p)*100:.1f}")
    RESULTS[f"baseline_{nm}"] = dict(acc=k / n, f1=float(f1_score(y, p)))

vd = pd.concat([va, te], ignore_index=True)
vd["pred"] = clf.predict(v.transform(vd.text))

print("\nPer sub-topik (val+dev_test; sel kecil -> baca interval, bukan titik):")
for t, g in vd.groupby("sub_topic"):
    h, f = g[g.label == 1], g[g.label == 0]
    print(f"  {t:<10} recall hoaks {fmt(int((h.pred == 1).sum()), len(h))} | spesifisitas fakta {fmt(int((f.pred == 0).sum()), len(f))}")

print("\nPer sumber (proporsi benar; val+dev_test):")
for s, g in vd.groupby("source"):
    print(f"  {s:<18} {fmt(int((g.pred == g.label).sum()), len(g))}")

val       akurasi  94.1% [90.3-96.4] (n=236) | F1 hoaks 93.9
dev_test  akurasi  91.5% [87.3-94.4] (n=236) | F1 hoaks 91.1

Per sub-topik (val+dev_test; sel kecil -> baca interval, bukan titik):
  hukum      recall hoaks  94.4% [74.2-99.0] (n=18) | spesifisitas fakta  91.7% [64.6-98.5] (n=12)
  kebijakan  recall hoaks  85.4% [71.6-93.1] (n=41) | spesifisitas fakta 100.0% [91.8-100.0] (n=43)
  pemilu     recall hoaks  85.3% [75.0-91.8] (n=68) | spesifisitas fakta  91.9% [82.5-96.5] (n=62)
  tokoh      recall hoaks  91.7% [85.0-95.6] (n=109) | spesifisitas fakta  98.3% [94.1-99.5] (n=119)

Per sumber (proporsi benar; val+dev_test):
  cnnindonesia.com    97.5% [91.4-99.3] (n=81)
  hoax_archive_15k    88.2% [76.6-94.5] (n=51)
  kompas.com          97.4% [92.6-99.1] (n=115)
  tempo.co            92.5% [80.1-97.4] (n=40)
  turnbackhoax.id     89.2% [83.9-92.9] (n=185)


# 5: Ablasi Masukan (Input Ablation — Di Mana Sinyalnya?)

In [6]:
first8 = lambda s: " ".join(str(s).split()[:8])
nb_tr, nb_va = tr.body.str.len() > 0, va.body.str.len() > 0

VAR = {
    "teks penuh":      (tr.text, ytr, va.text, yva),
    "judul saja":      (tr.title.map(prep_title), ytr, va.title.map(prep_title), yva),
    "isi saja":        (tr.body[nb_tr], ytr[nb_tr.to_numpy()], va.body[nb_va], yva[nb_va.to_numpy()]),
    "8 kata pertama":  (tr.text.map(first8), ytr, va.text.map(first8), yva),
}

for nm, (Xa, ya, Xb, yb) in VAR.items():
    _, _, p = run(Xa, ya, Xb); k = int((p == yb).sum())
    print(f"{nm:<15} val {fmt(k, len(yb))}")
    RESULTS[f"ablasi_{nm}"] = k / len(yb)

print("Galat baku ±3-4 poin pada n≈236; selisih kecil antar-varian tidak bermakna.")

teks penuh      val  94.1% [90.3-96.4] (n=236)
judul saja      val  88.1% [83.4-91.7] (n=236)
isi saja        val  98.3% [95.7-99.3] (n=236)
8 kata pertama  val  83.5% [78.2-87.7] (n=236)
Galat baku ±3-4 poin pada n≈236; selisih kecil antar-varian tidak bermakna.


# 6: Evaluasi Out-of-Distribution (OOD — Di Luar Distribusi)

In [7]:
nk = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
seen = set(tr.title.map(nk)) | set(va.title.map(nk)) | set(te.title.map(nk))
vt, mt, pt_ = run(tr.title.map(prep_title), ytr, va.title.map(prep_title))
print(f"Model judul-saja: val {fmt(int((pt_ == yva).sum()), len(yva))}")

def evaluasi(path, nama, harapan, filter_klaim=False):
    d = pd.read_csv(path)
    d["title"] = d.title.fillna("").astype(str)
    n0 = len(d)
    if filter_klaim:
        tag = d.title.str.match(r"(?i)^\s*\[(?:hoaks|hoax|salah|disinformasi)\]")
        if tag.sum() > 0: d = d[tag]
    d["k"] = d.title.map(nk)
    d = d.drop_duplicates("k"); d = d[~d.k.isin(seen)]
    X = d.title.map(prep_title)
    p = mt.predict(vt.transform(X))
    k = int(p.sum())
    print(f"{nama:<28} n awal={n0}, dipakai={len(d)} | diprediksi HOAKS: {fmt(k, len(d))}  <- {harapan}")
    RESULTS[nama] = dict(n=len(d), share_hoaks=k / max(len(d), 1))
    return d.assign(pred=p)

eb = evaluasi(f"{BASE}/eval_benar.csv", "eval_benar", "makin tinggi = model menilai gaya Mafindo, bukan isi")
ek = evaluasi(f"{BASE}/eval_kompas_cekfakta.csv", "eval_kompas", "makin tinggi = mengenali hoaks non-Mafindo", filter_klaim=True)
print("\nCatatan: kedua set kecil, interval lebar. Teks judul-saja lebih pendek dari data latih (10-60 kata).")

Model judul-saja: val  88.1% [83.4-91.7] (n=236)
eval_benar                   n awal=432, dipakai=432 | diprediksi HOAKS:  83.8% [80.0-87.0] (n=432)  <- makin tinggi = model menilai gaya Mafindo, bukan isi
eval_kompas                  n awal=40, dipakai=20 | diprediksi HOAKS:  85.0% [64.0-94.8] (n=20)  <- makin tinggi = mengenali hoaks non-Mafindo

Catatan: kedua set kecil, interval lebar. Teks judul-saja lebih pendek dari data latih (10-60 kata).


# 7 Simpan Hasil Baseline ke JSON

In [8]:
RESULTS["pipeline"] = tp.PIPELINE_VERSION
with open("hasil_baseline.json", "w") as f:
    json.dump(RESULTS, f, indent=2)

print("HASIL LENGKAP TERSIMPAN DI hasil_baseline.json:")
print(json.dumps(RESULTS, indent=2))

HASIL LENGKAP TERSIMPAN DI hasil_baseline.json:
{
  "permutasi": {
    "mean": 49.42796610169491,
    "sd": 3.742931190698685,
    "max": 55.932203389830505
  },
  "baseline_val": {
    "acc": 0.940677966101695,
    "f1": 0.9391304347826087
  },
  "baseline_dev_test": {
    "acc": 0.9152542372881356,
    "f1": 0.9107142857142857
  },
  "ablasi_teks penuh": 0.940677966101695,
  "ablasi_judul saja": 0.8813559322033898,
  "ablasi_isi saja": 0.9830508474576272,
  "ablasi_8 kata pertama": 0.8347457627118644,
  "eval_benar": {
    "n": 432,
    "share_hoaks": 0.8379629629629629
  },
  "eval_kompas": {
    "n": 20,
    "share_hoaks": 0.85
  },
  "pipeline": "tp-1.6"
}


# 8 AUDIT KRITIS & SANITY CHECK (PRA-COLAB 02)

In [9]:
# %% [8] AUDIT KRITIS & SANITY CHECK (PRA-COLAB 02) ------------------------------
import subprocess
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier

print("=" * 80)
print("1. CEK INTEGRITAS ZIP & KEBOCORAN TEKS ANTAR-SPLIT")
print("=" * 80)
# Cek SHA-256 ZIP & pastikan lockbox tidak ada di dalam zip
!sha256sum /content/dataset_v2_frozen.zip
!unzip -l /content/dataset_v2_frozen.zip | grep -i lockbox || echo "LOCKBOX TERBUKTI TIDAK ADA DI ZIP"

# Cek kemiripan teks menggunakan Cosine Similarity
v0 = TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True).fit(pd.concat([tr.text, va.text, te.text]))
Xt = v0.transform(tr.text)
for nm, d in (("val", va), ("dev_test", te)):
    s = (v0.transform(d.text) @ Xt.T).max(axis=1).toarray().ravel()
    print(f"Kemiripan {nm:<9} ke train -> cos > 0.7: {int((s > 0.7).sum())} | cos > 0.9: {int((s > 0.9).sum())}")

# Cek irisan group_id
print("Irisan group_id lintas split (tr∩va, tr∩te, va∩te):",
      len(set(tr.group_id) & set(va.group_id)),
      len(set(tr.group_id) & set(te.group_id)),
      len(set(va.group_id) & set(te.group_id)))

print("\n" + "=" * 80)
print("2. UJI BIAS PANJANG TEKS PADA 'ISI SAJA'")
print("=" * 80)
# Evaluasi Decision Tree pada panjang kata murni
w = lambda d: d.body.str.split().str.len().to_frame("w")
dt_score = DecisionTreeClassifier(max_depth=3, random_state=0).fit(w(tr), ytr).score(w(va), yva)
print(f"Akurasi model HANYA dari panjang kata body: {dt_score*100:.1f}%")

# Ablasi isi dipotong 50 kata (adil terhadap batasan pipeline)
cap50 = lambda s: tp.cap_sentences(str(s), 50)
nb_tr, nb_va = tr.body.str.len() > 0, va.body.str.len() > 0
_, _, p_cap = run(tr.body[nb_tr].map(cap50), ytr[nb_tr.to_numpy()], va.body[nb_va].map(cap50))
print(f"Akurasi isi saja (dipotong 50 kata)      : {fmt(int((p_cap == yva[nb_va.to_numpy()]).sum()), len(yva[nb_va.to_numpy()]))}")

print("\n" + "=" * 80)
print("3. VALIDASI OOD (EVAL_BENAR & RECALL HOAKS VAL)")
print("=" * 80)
yh = (yva == 1)
recall_val_hoaks = (mt.predict(vt.transform(va.title[yh].map(prep_title))) == 1).sum()
print(f"Recall hoaks val (judul saja)            : {fmt(int(recall_val_hoaks), int(yh.sum()))}")
print("\nContoh 8 judul eval_benar setelah prep_title (pastikan tag [BENAR] hilang):")
print(eb.title.head(8).map(prep_title).tolist())

print("\n" + "=" * 80)
print("4. VERIFIKASI REGEX EVAL_KOMPAS_CEKFAKTA")
print("=" * 80)
rk = pd.read_csv(f"{BASE}/eval_kompas_cekfakta.csv")
tg = rk.title.astype(str).str.match(r"(?i)^\s*\[(?:hoaks|hoax|salah|disinformasi)\]")
print(f"Tag cocok: {int(tg.sum())} dari total {len(rk)}")
print("Contoh judul yang TIDAK lolos filter tag (dibuang):")
print(rk.title[~tg].head(5).tolist())

print("\n" + "=" * 80)
print("5. SPESIFIKASI LINGKUNGAN KOMPUTASI (UNTUK TABEL 3.2 SKRIPSI)")
print("=" * 80)
import torch, transformers
print(f"PyTorch Version     : {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"GPU Akselerator     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
!free -h | awk '/Mem:/ {print "RAM Sistem          : " $2}'
!df -h / | awk 'NR==2 {print "Ruang Disk Tersedia : " $4}'
print("=" * 80)

1. CEK INTEGRITAS ZIP & KEBOCORAN TEKS ANTAR-SPLIT
011bd78465983f386074d269aae1671014f6554acfc2da574a3902bdb68b487a  /content/dataset_v2_frozen.zip
LOCKBOX TERBUKTI TIDAK ADA DI ZIP
Kemiripan val       ke train -> cos > 0.7: 0 | cos > 0.9: 0
Kemiripan dev_test  ke train -> cos > 0.7: 3 | cos > 0.9: 0
Irisan group_id lintas split (tr∩va, tr∩te, va∩te): 0 0 0

2. UJI BIAS PANJANG TEKS PADA 'ISI SAJA'
Akurasi model HANYA dari panjang kata body: 94.1%
Akurasi isi saja (dipotong 50 kata)      :  93.6% [89.8-96.1] (n=236)

3. VALIDASI OOD (EVAL_BENAR & RECALL HOAKS VAL)
Recall hoaks val (judul saja)            :  92.4% [86.1-95.9] (n=118)

Contoh 8 judul eval_benar setelah prep_title (pastikan tag [BENAR] hilang):
['Bahaya Daging Kurban Dibungkus Kresek Hitam.', 'SPP SMA/SMK GRATIS JAWA TIMUR.', 'Klarifikasi BIN terkait Pesan Berantai yang Sebut 29 Tokoh Pendukung Prabowo-Sandiaga sedang Dimonitori oleh Aparat.', 'Dianggap Lecehkan Guru, Iklan Hago Diprotes.', 'Klarifikasi “#SOSDrJohnSimpson

# 9 CEK KUALITATIF TAMBAHAN DARI REVIEWER

In [10]:
# (a) Intip 3 pasangan dev_test -> train yang mirip (cos > 0.7)
print("=" * 80)
print("1. TEKS PASANGAN DEV_TEST -> TRAIN (cos > 0.7):")
print("=" * 80)
from sklearn.feature_extraction.text import TfidfVectorizer
v2 = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.5, sublinear_tf=True).fit(pd.concat([tr.text, va.text, te.text]))
S = (v2.transform(te.text) @ v2.transform(tr.text).T).tocoo()
m = S.data > 0.7
if m.sum() == 0:
    print("Tidak ada pasangan dengan kemiripan > 0.7 pada konfigurasi TF-IDF v2 ini.")
for i, j, c in zip(S.row[m], S.col[m], S.data[m]):
    print(f"Skor Cosine: {round(float(c), 3)}")
    print("DEV  :", te.text[i][:220])
    print("TRAIN:", tr.text[j][:220])
    print("-" * 50)

# (b) Bedah komposisi kata 'Klarifikasi' di eval_benar
print("\n" + "=" * 80)
print("2. KOMPOSISI EVAL_BENAR (JUDUL DENGAN 'KLARIFIKASI' VS LAINNYA):")
print("=" * 80)
eb["t"] = eb.title.map(prep_title)
eb["klarifikasi"] = eb.t.str.match(r"(?i)^klarifikasi")
print(eb.groupby("klarifikasi").pred.agg(["sum", "size", "mean"]).round(3))

print("\nContoh 15 Judul eval_benar TANPA kata 'Klarifikasi':")
non_klarifikasi = eb[~eb.klarifikasi].t
print(non_klarifikasi.sample(min(15, len(non_klarifikasi)), random_state=0).to_string())

1. TEKS PASANGAN DEV_TEST -> TRAIN (cos > 0.7):
Skor Cosine: 0.707
DEV  : Saat Apel Siaga, Yaqut Sebut Banser Saudara Tua TNI-Polri, tapi Tidak Gajian. Ketua Umum Gerakan Pemuda (GP) Ansor Yaqut Cholil Qoumas memimpin apel siaga Ansor dan Banser di Alun-alun Kidul Yogyakarta, Minggu sore 23 Ok
TRAIN: Saat Yaqut Singgung Soal Kualat Kiai di Depan Ribuan Kader Ansor. Ketua Umum Gerakan Pemuda (GP) Ansor Yaqut Cholil Qoumas menyinggung soal kualat kiai saat menghadiri apel siaga Ansor dan Banser di Alun-alun Kidul Yogya
--------------------------------------------------
Skor Cosine: 0.719
DEV  : Semua Aktifitas HP Terpantau BSSN. Semua aktifitas Hand Phone n Media Telekomunikasi Serta segala Media Sosial terpantau 100%Setelah dilantiknya Badan Siber & Sandi Nasional (BSSN),17 Mei 2018 , oleh Bpk Jokowi,Presiden 
TRAIN: Pemantauan Aktivitas Telepon Seluler dan Media Sosial Oleh BSSN. Semua aktifitas HP dll .terpantau 100% Mulai besok dan seterusnya ada peraturan komunikasi baru. Setelah di

# 10 UJI BACA-SAJA: DIKSI SANGGAHAN PADA DATA FAKTA

In [11]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, cross_val_predict

al = pd.concat([tr, va, te], ignore_index=True)
X = al.title.map(prep_title)
pipe = make_pipeline(TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True), LogisticRegression(max_iter=2000))
al["oof"] = cross_val_predict(pipe, X, al.label, groups=al.group_id, cv=GroupKFold(5))

# Deteksi judul berita fakta yang memuat kata bantahan/sanggahan
rb = al.title.str.contains(r"(?i)\b(?:bantah|membantah|klarifikasi|mengklarifikasi|tepis|tidak ada|tidak benar)\b")

print("=" * 80)
print("HASIL UJI DIKSI SANGGAHAN (OOF 5-FOLD):")
print("=" * 80)
print("Fakta berdiksi sanggahan, diprediksi hoaks:", al[(al.label==0) & rb].oof.agg(["sum","size","mean"]).round(3).to_dict())
print("Fakta TANPA diksi itu, diprediksi hoaks   :", round(al[(al.label==0) & ~rb].oof.mean(), 3))
print("Hoaks keseluruhan, diprediksi hoaks       :", round(al[al.label==1].oof.mean(), 3))
print("=" * 80)

HASIL UJI DIKSI SANGGAHAN (OOF 5-FOLD):
Fakta berdiksi sanggahan, diprediksi hoaks: {'sum': 1.0, 'size': 12.0, 'mean': 0.083}
Fakta TANPA diksi itu, diprediksi hoaks   : 0.163
Hoaks keseluruhan, diprediksi hoaks       : 0.903


# 11 PERBAIKAN HASIL_BASELINE.JSON RESMI

In [12]:
RESULTS["ablasi_isi saja"] = (
    "TIDAK VALID (body tidak dipotong, terkonfus panjang); pakai"
    " ablasi_isi_saja_50_kata"
)
RESULTS["catatan_tambahan"] = {
    "ablasi_isi_saja_50_kata": 0.936,
    "pohon_panjang_body_saja": 0.941,
    "pasangan_dev_ke_train_cos07": (
        3
    ),  # 1 varian BSSN, 1 acara sama, 1 templat scam
    "oof_fakta_diksi_sanggahan_diprediksi_hoaks": [1, 12],
    "oof_fakta_lain_diprediksi_hoaks": 0.163,
    "oof_hoaks_diprediksi_hoaks": 0.903,
    "eval_benar_klarifikasi_vs_lain": [0.862, 0.830],
}

with open("hasil_baseline.json", "w") as f:
  json.dump(RESULTS, f, indent=2)

from google.colab import files

files.download("hasil_baseline.json")
print("hasil_baseline.json resmi berhasil diperbarui dan diunduh!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

hasil_baseline.json resmi berhasil diperbarui dan diunduh!
